# Bonus Ablation — Model L (Nepali) Kaggle Notebook

Trains and evaluates the assignment's **required** Bonus-section ablation on Model L (Nepali): positional embeddings removed entirely, compared against the standard Phase 2 model.

In [36]:
VARIANT = "no_pos"  # the required Bonus-section ablation: positional embeddings removed
print("Running variant:", VARIANT)

Running variant: no_pos


## Locate the attached dataset
Just the code + Nepali data -- the baseline checkpoint comes from Drive instead (see below).

In [37]:
import os
from pathlib import Path

def find_root(base="/kaggle/input", marker="train_ablation.py"):
    for root, dirs, files in os.walk(base):
        if marker in files:
            return Path(root)
    raise FileNotFoundError(
        f"Couldn't find a folder containing '{marker}' under {base}. "
        "Run `!find /kaggle/input -maxdepth 4` to inspect what got mounted, "
        "and confirm the ablation dataset is attached."
    )

ABLATION_INPUT = find_root()
print("Ablation bundle found at:", ABLATION_INPUT)
os.system(f"du -sh {ABLATION_INPUT}")

Ablation bundle found at: /kaggle/input/datasets/ervarishitha11/ablation-phase2/ablation
6.4G	/kaggle/input/datasets/ervarishitha11/ablation-phase2/ablation


0

## Build a writable working copy
Copies only the small code/config files (so `outputs/` is writable). The large read-only `NEPALI/` data splits (~6GB) are symlinked straight out of the mounted dataset instead of copied, so this stays fast and doesn't eat into `/kaggle/working`'s disk quota. `baseline_checkpoint/` is created empty here -- the Drive-pull cell below fills it.

In [38]:
import shutil

WORK = Path("/kaggle/working/ablation")
if WORK.exists():
    shutil.rmtree(WORK)
WORK.mkdir(parents=True)

shutil.copytree(ABLATION_INPUT / "src", WORK / "src")
shutil.copytree(ABLATION_INPUT / "configs", WORK / "configs")
os.symlink(ABLATION_INPUT / "tokenizer", WORK / "tokenizer")
os.symlink(ABLATION_INPUT / "NEPALI", WORK / "NEPALI")
for s in ["train_ablation.py", "eval_ablation.py"]:
    shutil.copy(ABLATION_INPUT / s, WORK / s)

(WORK / "outputs").mkdir(parents=True, exist_ok=True)
(WORK / "baseline_checkpoint").mkdir(parents=True, exist_ok=True)  # filled by the Drive-pull cell below

print("Ready:")
os.system(f"find {WORK} -maxdepth 3 | sort")

Ready:
/kaggle/working/ablation
/kaggle/working/ablation/baseline_checkpoint
/kaggle/working/ablation/configs
/kaggle/working/ablation/configs/model_l_no_pos_config.json
/kaggle/working/ablation/configs/model_l_rmsnorm_config.json
/kaggle/working/ablation/eval_ablation.py
/kaggle/working/ablation/NEPALI
/kaggle/working/ablation/outputs
/kaggle/working/ablation/src
/kaggle/working/ablation/src/attention_analyzer.py
/kaggle/working/ablation/src/dataset.py
/kaggle/working/ablation/src/__init__.py
/kaggle/working/ablation/src/metrics.py
/kaggle/working/ablation/src/model.py
/kaggle/working/ablation/src/trainer.py
/kaggle/working/ablation/tokenizer
/kaggle/working/ablation/train_ablation.py


sh: 0: getcwd() failed: No such file or directory


0

## Install dependencies
(`sentencepiece`/`sacrebleu`/`rouge-score` aren't preinstalled on Kaggle; everything else -- torch, matplotlib, tqdm -- already is.)

In [39]:
os.system("pip install -q sentencepiece sacrebleu rouge-score")

sh: 0: getcwd() failed: No such file or directory
shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
The folder you are executing pip from can no longer be found.


256

## Connect Google Drive (rclone) — required for the baseline comparison
The standard Model L checkpoint lives at `gdrive:2025201027_LMA_Phase2/NEPALI_checkpoints/checkpoint_final.pt` and is pulled down below, using the same `RCLONE_CONF` secret as the Phase 2 notebooks (Kaggle's Secret value field flattens real newlines into spaces, so the config is stored with newlines encoded as literal `\n` and decoded back below). If `USE_RCLONE = False`, training still works, but `eval_ablation.py` skips the standard-model comparison since it has nothing to compare against.

In [40]:
USE_RCLONE = True  # set False to skip Drive sync entirely

if USE_RCLONE:
    os.system("curl https://rclone.org/install.sh | sudo bash")
    from kaggle_secrets import UserSecretsClient
    conf_dir = os.path.expanduser("~/.config/rclone")
    os.makedirs(conf_dir, exist_ok=True)
    raw_secret = UserSecretsClient().get_secret("RCLONE_CONF")
    conf_content = raw_secret.replace("\\n", "\n")
    with open(f"{conf_dir}/rclone.conf", "w") as f:
        f.write(conf_content)
    os.system("rclone lsd gdrive:")  # sanity check -- should list your Drive's top-level folders
else:
    print("Skipping rclone setup -- remember to Save Version before this session ends.")

sh: 0: getcwd() failed: No such file or directory
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0shell-init: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
100  4734  100  4734    0     0  11818      0 --:--:-- --:--:-- --:--:-- 11835
job-working-directory: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory
chdir: error retrieving current directory: getcwd: cannot access parent directories: No such file or directory



The latest version of rclone rclone v1.75.1 is already installed.



sh: 0: getcwd() failed: No such file or directory
2026/09/07 07:08:05 NOTICE: gdrive: This remote uses rclone's shared Google Drive client_id, which is being retired and will stop working during 2026. Create your own client_id to avoid interruption: https://rclone.org/drive/#making-your-own-client-id


           0 2026-08-21 05:27:42        -1 2025201027_LMA_Phase1
           0 2026-09-05 04:35:49        -1 2025201027_LMA_Phase2
           0 2026-08-16 08:50:01        -1 Colab Notebooks
           0 2026-09-05 11:41:32        -1 LMA_Project1


## Pull the standard model's checkpoint from Drive (for comparison)
No-op-safe to re-run. If this fails (wrong path, no access), the ablation still trains and evaluates fine -- `eval_ablation.py` just skips the standard-vs-ablated comparison table and warns instead of erroring.

In [41]:
if USE_RCLONE:
    os.system(f"rclone copy gdrive:2025201027_LMA_Phase2/NEPALI_checkpoints/checkpoint_final.pt {WORK}/baseline_checkpoint/ -P")
    os.system(f"ls -la {WORK}/baseline_checkpoint/")
else:
    print("Skipped -- no standard-model checkpoint pulled, comparison table will be skipped later.")

sh: 0: getcwd() failed: No such file or directory
2026/09/07 07:08:22 NOTICE: gdrive: This remote uses rclone's shared Google Drive client_id, which is being retired and will stop working during 2026. Create your own client_id to avoid interruption: https://rclone.org/drive/#making-your-own-client-id


Transferred:   	          0 B / 295.880 MiB, 0%, 0 B/s, ETA -
Transferred:            0 / 1, 0%
Elapsed time:         0.3s
Transferring:
 *                           checkpoint_final.pt:  0% / 295.880 MiB, 0 B/s, -Transferred:   	    5.125 MiB / 295.880 MiB, 2%, 0 B/s, ETA -
Transferred:            0 / 1, 0%
Elapsed time:         0.8s
Transferring:
 *                           checkpoint_final.pt:  1% / 295.880 MiB, 0 B/s, -Transferred:   	  105.500 MiB / 295.880 MiB, 36%, 22.373 MiB/s, ETA 8s
Transferred:            0 / 1, 0%
Elapsed time:         1.3s
Transferring:
 *                           checkpoint_final.pt: 35% / 295.880 MiB, 22.370 MiB/Transferred:   	  201.875 MiB / 295.880 MiB, 68%, 22.373 MiB/s, ETA 4s
Transferred:            0 / 1, 0%
Elapsed time:         1.8s
Transferring:
 *                           checkpoint_final.pt: 68% / 295.880 MiB, 22.370 MiB/Transferred:   	      228 MiB / 295.880 MiB, 77%, 103.999 MiB/s, ETA 0s
Transferred:            0 / 1, 0%
Elapsed time: 

sh: 0: getcwd() failed: No such file or directory


## Optional: pull this variant's checkpoint back down
No-op the first time. If a previous session already pushed `checkpoint_latest.pt` for this variant to Drive, this resumes training from that step instead of from scratch.

In [42]:
if USE_RCLONE:
    os.system(f"rclone copy gdrive:LMA_Project1/ablation_checkpoints/{VARIANT} {WORK}/outputs/{VARIANT}/checkpoints -P")

sh: 0: getcwd() failed: No such file or directory
2026/09/07 07:08:32 NOTICE: gdrive: This remote uses rclone's shared Google Drive client_id, which is being retired and will stop working during 2026. Create your own client_id to avoid interruption: https://rclone.org/drive/#making-your-own-client-id


Transferred:   	          0 B / 1.145 GiB, 0%, 0 B/s, ETA -
Checks:                 0 / 0, -, Listed 18
Transferred:            0 / 16, 0%
Elapsed time:         0.5s
Transferring:
 *                           checkpoint_final.pt:  0% / 292.879 MiB, 0 B/s, -
 *                          checkpoint_latest.pt:  0% / 292.879 MiB, 0 B/s, -
 *                      checkpoint_step_25000.pt:  0% / 292.880 MiB, 0 B/s, -
 *                      checkpoint_step_30000.pt:  0% / 292.880 MiB, 0 B/s, -Transferred:   	  126.875 MiB / 1.145 GiB, 11%, 0 B/s, ETA -
Checks:                 0 / 0, -, Listed 18
Transferred:            0 / 16, 0%
Elapsed time:         1.0s
Transferring:
 *                           checkpoint_final.pt: 10% / 292.879 MiB, 0 B/s, -
 *                          checkpoint_latest.pt: 17% / 292.879 MiB, 0 B/s, -
 *                      checkpoint_step_25000.pt: 10% / 292.880 MiB, 0 B/s, -
 *                      checkpoint_step_30000.pt:  4% / 292.880 MiB, 0 B/s, -Transferred:   	 

## Optional: start the background Drive-sync loop
Syncs this variant's checkpoints every 5 minutes for the life of the session.

In [43]:
if USE_RCLONE:
    sync_sh = f"""
while true; do
  rclone sync {WORK}/outputs/{VARIANT}/checkpoints gdrive:LMA_Project1/ablation_checkpoints/{VARIANT} --include \"checkpoint_latest.pt\" --include \"checkpoint_final.pt\" --include \"checkpoint_step_*.pt\" --include \"training_history.json\"
  sleep 300
done
"""
    open("/kaggle/working/sync_ablation.sh", "w").write(sync_sh)
    os.system("nohup bash /kaggle/working/sync_ablation.sh > /kaggle/working/sync_ablation.log 2>&1 &")
    print("Background Drive sync started for variant:", VARIANT)

Background Drive sync started for variant: no_pos


sh: 0: getcwd() failed: No such file or directory


## Sanity check before spending GPU hours
Confirms the GPU is visible, the ablated model builds with the right parameter count, and the causal-mask check still passes.

In [44]:
import torch, json, sys
print("CUDA available:", torch.cuda.is_available(), "-",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")

os.chdir(str(WORK))
sys.path.insert(0, str(WORK))
from src.model import GPTLanguageModel
from src.dataset import SentencePieceTokenizer

config = json.load(open(WORK / "configs" / f"model_l_{VARIANT}_config.json"))
model = GPTLanguageModel(
    vocab_size=config["vocab_size"], d_model=config["d_model"], n_layer=config["n_layer"],
    n_head=config["n_head"], d_ff=config["d_ff"], max_seq_len=config["max_seq_len"],
    dropout=config["dropout"], tie_weights=config["tie_weights"],
    use_positional_embeddings=config["use_positional_embeddings"], norm_type=config["norm_type"]
)
params = model.count_parameters()
print("Total trainable parameters:", f"{params['total_trainable']:,}")
print("use_positional_embeddings:", params["use_positional_embeddings"], "| norm_type:", params["norm_type"])
assert 24_000_000 < params["total_trainable"] < 26_000_000
assert model.verify_causal_masking(), "causal mask check failed"
print("Causal masking check: PASSED")

tok = SentencePieceTokenizer(str(WORK / "tokenizer" / "nepali_bpe.model"))
print("Tokenizer vocab size:", tok.vocab_size)
print("Sample roundtrip:", tok.decode(tok.encode("नेपाल एक देश हो")))
print("\nALL SANITY CHECKS PASSED")

CUDA available: True - Tesla T4
Total trainable parameters: 25,059,328
use_positional_embeddings: False | norm_type: layernorm
Causal masking check: PASSED
Tokenizer vocab size: 12000
Sample roundtrip: नेपाल एक देश हो

ALL SANITY CHECKS PASSED


## Pretrain the ablated model
Auto-resumes from `checkpoint_latest.pt` if one was pulled down above.

In [9]:
%cd {WORK}
!python train_ablation.py --variant {VARIANT}

/kaggle/working
=== Starting Ablation Pretraining: variant='no_pos' ===
Ablation: Required Bonus ablation: positional embeddings removed entirely. Same data, tokenizer, and hyperparameters as the standard Model L run in NEPALI/PHASE2/ -- only this flag differs.
Config: {'language': 'nepali', 'model_tier': 'Model L', 'ablation_type': 'no_positional_embeddings', 'ablation_description': 'Required Bonus ablation: positional embeddings removed entirely. Same data, tokenizer, and hyperparameters as the standard Model L run in NEPALI/PHASE2/ -- only this flag differs.', 'use_positional_embeddings': False, 'norm_type': 'layernorm', 'vocab_size': 12000, 'd_model': 512, 'n_layer': 6, 'n_head': 8, 'd_ff': 2048, 'max_seq_len': 512, 'dropout': 0.1, 'tie_weights': True, 'learning_rate': 0.0006, 'weight_decay': 0.1, 'betas': [0.9, 0.95], 'batch_size': 32, 'max_steps': 30500, 'warmup_steps': 1000, 'grad_clip': 1.0, 'eval_interval': 500, 'save_interval': 1000, 'milestone_interval': 5000, 'keep_last_n_c

## Evaluate: ablated model + standard model side by side
Run once `checkpoint_final.pt` exists (training reached `max_steps`), or any time for an interim eval against `checkpoint_latest.pt`. Uses the standard-model checkpoint pulled from Drive above to write the comparison table -- if that pull didn't happen or failed, this still runs fine and just skips the comparison (see its printed warning).

In [45]:
!python eval_ablation.py --variant {VARIANT}

if USE_RCLONE:
    os.system(f"rclone sync {WORK}/outputs/{VARIANT}/checkpoints gdrive:LMA_Project1/ablation_checkpoints/{VARIANT} -P")
    os.system(f"rclone copy {WORK}/outputs/{VARIANT}/results gdrive:LMA_Project1/ablation_checkpoints/{VARIANT}/results -P")
    os.system(f"rclone copy {WORK}/outputs/{VARIANT}/plots gdrive:LMA_Project1/ablation_checkpoints/{VARIANT}/plots -P")
    print("Done -- ablation checkpoint, results, and plots are all on Drive.")
else:
    print("Done -- remember to Save Version to keep these outputs.")

=== Evaluating Ablation 'no_pos' (+ standard Model L for comparison) on cuda ===
Loaded ablated checkpoint: /kaggle/working/ablation/outputs/no_pos/checkpoints/checkpoint_final.pt
[TextDataset] Cached 5,932 docs / 2,688,965 tokens -> val.v12000.dall.ids.bin (10.8 MB)

--- [ablated_no_pos] Intrinsic LM Metrics (Validation) ---
Val Loss:   3.5969
Perplexity: 36.4844
BPB:        0.4385
[TextDataset] Cached 5,932 docs / 2,600,959 tokens -> test.v12000.dall.ids.bin (10.4 MB)

--- [ablated_no_pos] Intrinsic LM Metrics (Test) ---
Test Loss:  3.5834
Test PPL:   35.9974
Test BPB:   0.4359

[ablated_no_pos] Prompt: प्राईड नेपालको आयोजनामा प्रस्तुत “नव वर्ष 2073 साँस्कृतिक साँझ”
  [Temp 0.5]: प्राईड नेपालको आयोजनामा प्रस्तुत “नव वर्ष 2073 साँस्कृतिक साँझ”मा सम्पन्न भएको हो। प्राईड नेपालको आयोजनामा भएको उक्त कार्यक्रममा प्राई
  [Temp 1.0]: प्राईड नेपालको आयोजनामा प्रस्तुत “नव वर्ष 2073 साँस्कृतिक साँझ” घोषणा काठमाडौं । राष्ट्रिय लोक तथा दोहोरी साँझ भाषणा सांस्कृतिक साँझ राप्रपा
  [Temp 1.5]: प्राई

2026/09/07 07:13:11 NOTICE: gdrive: This remote uses rclone's shared Google Drive client_id, which is being retired and will stop working during 2026. Create your own client_id to avoid interruption: https://rclone.org/drive/#making-your-own-client-id


Transferred:   	          0 B / 585.760 MiB, 0%, 0 B/s, ETA -
Checks:                14 / 14, 100%, Listed 34
Transferred:            0 / 2, 0%
Elapsed time:         0.5s
Transferring:
 *                      checkpoint_step_25000.pt:  0% / 292.880 MiB, 0 B/s, -
 *                      checkpoint_step_30000.pt:  0% / 292.880 MiB, 0 B/s, -Transferred:   	       32 MiB / 585.760 MiB, 5%, 0 B/s, ETA -
Checks:                14 / 14, 100%, Listed 34
Transferred:            0 / 2, 0%
Elapsed time:         1.0s
Transferring:
 *                      checkpoint_step_25000.pt:  5% / 292.880 MiB, 0 B/s, -
 *                      checkpoint_step_30000.pt:  5% / 292.880 MiB, 0 B/s, -Transferred:   	       64 MiB / 585.760 MiB, 11%, 39.983 MiB/s, ETA 13s
Checks:                14 / 14, 100%, Listed 34
Transferred:            0 / 2, 0%
Elapsed time:         1.4s
Transferring:
 *                      checkpoint_step_25000.pt: 10% / 292.880 MiB, 23.991 MiB/
 *                      checkpoint_step_3000

2026/09/07 07:13:39 NOTICE: gdrive: This remote uses rclone's shared Google Drive client_id, which is being retired and will stop working during 2026. Create your own client_id to avoid interruption: https://rclone.org/drive/#making-your-own-client-id


Transferred:   	  139.835 KiB / 139.835 KiB, 100%, 0 B/s, ETA -
Checks:                 3 / 3, 100%, Listed 6
Transferred:            0 / 3, 0%
Elapsed time:         0.5s
Transferring:
 *                      eval_results_no_pos.json:100% / 70.181 KiB, 0 B/s, -
 *                    eval_results_standard.json:100% / 69.057 KiB, 0 B/s, -
 *              comparison_no_pos_vs_standard.md:100% / 612 B, 0 B/s, -Transferred:   	  139.835 KiB / 139.835 KiB, 100%, 0 B/s, ETA -
Checks:                 3 / 3, 100%, Listed 6
Transferred:            0 / 3, 0%
Elapsed time:         1.0s
Transferring:
 *                      eval_results_no_pos.json:100% / 70.181 KiB, 0 B/s, -
 *                    eval_results_standard.json:100% / 69.057 KiB, 0 B/s, -
 *              comparison_no_pos_vs_standard.md:100% / 612 B, 0 B/s, -Transferred:   	  139.835 KiB / 139.835 KiB, 100%, 139.823 KiB/s, ETA 0s
Checks:                 3 / 3, 100%, Listed 6
Transferred:            2 / 3, 67%
Elapsed time:         1.5s

2026/09/07 07:13:41 NOTICE: gdrive: This remote uses rclone's shared Google Drive client_id, which is being retired and will stop working during 2026. Create your own client_id to avoid interruption: https://rclone.org/drive/#making-your-own-client-id


Transferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Checks:                 0 / 8, 0%, Listed 16
Elapsed time:         0.5s
Checking:
 *  nepali_no_pos_attn_layer1_multihead_grid.png: checking
 *           nepali_no_pos_attn_layer1_head1.png: checking
 *           nepali_no_pos_attn_layer6_head1.png: checking
 *  nepali_no_pos_attn_layer6_multihead_grid.png: checking
 *         nepali_standard_attn_layer1_head1.png: checking
 * nepali_standard_attn_l…er1_multihead_grid.png: checking
 *         nepali_standard_attn_layer6_head1.png: checking
 * nepali_standard_attn_l…er6_multihead_grid.png: checkingTransferred:   	          0 B / 0 B, -, 0 B/s, ETA -
Checks:                 8 / 8, 100%, Listed 16
Elapsed time:         0.7s
Done -- ablation checkpoint, results, and plots are all on Drive.


## Results
Prints the standard-vs-ablated comparison table `eval_ablation.py` just wrote (perplexity/BPB, BLEU/chrF/ROUGE-L, diversity, attention entropy/distance -- standard vs. ablated), and lists the attention heatmaps saved alongside it. This is what goes into the report.

In [46]:
comparison_path = WORK / "outputs" / VARIANT / "results" / f"comparison_{VARIANT}_vs_standard.md"
print(open(comparison_path).read())

plots_dir = WORK / "outputs" / VARIANT / "plots"
print("\nAttention heatmaps saved:")
os.system(f"ls -la {plots_dir}")

# Ablation Comparison: Standard Model L vs. `no_pos`

| Metric | Standard Model L | Ablated (`no_pos`) |
|---|---|---|
| Val Loss | 3.5140 | 3.5969 |
| Val Perplexity | 33.5828 | 36.4844 |
| Val BPB | 0.4284 | 0.4385 |
| Test Loss | 3.4991 | 3.5834 |
| Test Perplexity | 33.0859 | 35.9974 |
| Test BPB | 0.4256 | 0.4359 |
| BLEU-4 | 3.2100 | 2.2800 |
| chrF++ | 22.7300 | 21.8200 |
| ROUGE-L | 6.9400 | 5.7600 |
| Distinct-1 | 0.6201 | 0.5714 |
| Distinct-2 | 0.6647 | 0.6611 |
| 4-gram Repetition | 0.2829 | 0.2593 |
| Attention Entropy (L1H1) | 1.1077 | 0.5459 |
| Attention Distance (L1H1) | 1.9287 | 0.5623 |

Attention heatmaps saved:
total 916
drwxr-xr-x 2 root root   4096 Sep  7 07:13 .
drwxr-xr-x 5 root root   4096 Sep  7 07:08 ..
-rw-r--r-- 1 root root  77969 Sep  7 07:10 nepali_no_pos_attn_layer1_head1.png
-rw-r--r-- 1 root root 152810 Sep  7 07:10 nepali_no_pos_attn_layer1_multihead_grid.png
-rw-r--r-- 1 root root  77949 Sep  7 07:10 nepali_no_pos_attn_layer6_head1.png
-rw-r--r-- 1 

0